## Setup

This project uses NVIDIA’s Earth-2 (Earth2Studio) framework.

### Prerequisites

- NVIDIA Earth-2 (Earth2Studio)
- Python 3.12 kernel
- `uv` (recommended)


If you do not have `uv` installed, follow the installation instructions for your system:
https://docs.astral.sh/uv/getting-started/installation/

### Python kernel

To install the proper kernel, run ```uv run python -m ipykernel install --user --name earth2 --display-name "Earth2 (Python 3.12)"```

For most users, follow the official Earth-2 installation guide:
https://docs.nvidia.com/earth-2/


In [1]:
import earth2studio, inspect
from earth2studio.models.px import fcn, FCN

print("earth2studio version:", earth2studio.__version__)
print("fcn module path:", fcn.__file__)

src = inspect.getsource(fcn)
print("loader expects fcn.zip?:", "fcn.zip" in src)
print("loader expects fcn.mdlus?:", "fcn.mdlus" in src)

pkg = FCN.load_default_package()
print("default package root:", pkg.__dict__.get("root"))

/home/jovyan/earth2studio-project/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/jovyan/earth2studio-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CuPy distance computation test failed with error: cuVS >= 24.12 or pylibraft < 24.12 should be installed to use this feature


Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/home/jovyan/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x73463e386b00> is already entered


earth2studio version: 0.13.0a0
fcn module path: /home/jovyan/earth2studio-project/.venv/lib/python3.12/site-packages/earth2studio/models/px/fcn.py
loader expects fcn.zip?: False
loader expects fcn.mdlus?: True
default package root: hf://nvidia/fourcastnet1@c67a63995f6c8e0e557eb3d791f32f437e9b02d5


In [ ]:
from datetime import datetime
import earth2studio.run as run
from earth2studio.models.px import FCN
from earth2studio.models.px import FCN3
from earth2studio.data import GFS
from earth2studio.io import ZarrBackend
from earth2studio.models.dx import PrecipitationAFNO
import xarray as xr
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import os

In [2]:
os.chdir("earth2studio-project")

NameError: name 'os' is not defined

# Run the cell below to ensure you have enough GPU memory
If you need to kill a process, run ```kill -9 {PID}``` in your terminal


### 2m Air Temperature Forecast (T+24h)

In [ ]:
os.chdir("/home/jovyan/earth2studio-project")
model = FCN.load_model(FCN.load_default_package())
data = GFS()
io = ZarrBackend("t2m_outputs/fcn_gfs_forecast.zarr", backend_kwargs={'overwrite': True})

In [ ]:
nsteps = 10
io = run.deterministic(["2026-01-08"], nsteps, model, data, io)
print(io.root.tree())

In [ ]:
# plot t2m
def plot_prediction(ds, variable, step):
    plt.close("all")
    forecast = "2026-01-08 Prediction"
    # Create a Robinson projection
    projection = ccrs.Robinson()

    # Create a figure and axes with the specified projection
    fig, ax = plt.subplots(subplot_kw={"projection": projection}, figsize=(10, 6))

    # Plot the field using pcolormesh
    im = ax.pcolormesh(
        ds["lon"],
        ds["lat"],
        ds[variable][0, step],
        transform=ccrs.PlateCarree(),
        cmap="Spectral_r",
    )
    # temperature bar
    cbar = plt.colorbar(im, ax = ax, orientation = "horizontal", pad = 0.05)
    cbar.set_label("Temperature (K)")

    # Set title
    ax.set_title(f"{forecast} - Lead time: {6*step}hrs")

    # Add coastlines and gridlines
    ax.coastlines()
    ax.gridlines()
    plt.savefig("/home/jovyan/earth2studio-project/t2m_outputs/07_t2m_prediction.jpg")

In [ ]:
# plot t2m
ds = xr.open_zarr("/home/jovyan/earth2studio-project/t2m_outputs/fcn_gfs_forecast.zarr")
plot_prediction(ds, "t2m", 4)

### Total Precipitation Forecast

In [ ]:
prognostic_model = FCN.load_model(FCN.load_default_package())
package = PrecipitationAFNO.load_default_package()
diagnostic_model = PrecipitationAFNO.load_model(package)
io = ZarrBackend("/home/jovyan/earth2studio-project/tp_outputs/fcn_gfs_forecast.zarr", backend_kwargs={'overwrite': True})
data = GFS()

In [ ]:
nsteps = 10
io = run.diagnostic(
    ["2026-01-01"],
    nsteps,
    prognostic_model,
    diagnostic_model,
    data,
    io
)
print(io.root.tree())

In [ ]:
def plot_tp(ds, variable, step, extent):
    plt.close("all")
    forecast = datetime(2026, 1, 1)
    # Create a Orthographic projection of USA
    #projection = ccrs.Orthographic(-100, 40)
    projection = ccrs.Robinson()

    # Create a figure and axes with the specified projection
    fig, ax = plt.subplots(subplot_kw={"projection": projection}, figsize=(10, 6))

    # Plot the field using pcolormesh
    levels = np.arange(0.0, 0.01, 0.001)
    im = ax.contourf(
        ds["lon"],
        ds["lat"],
        ds[variable][0, step],
        levels,
        transform=ccrs.PlateCarree(),
        vmax=0.01,
        vmin=0.00,
        cmap="terrain",
        extend="max"
    )

    # Set title
    ax.set_title(f"{forecast.strftime('%Y-%m-%d')} - Lead time: {6*step}hrs")

    # Add coastlines and gridlines6
    ax.set_extent(extent)  # [lat min, lat max, lon min, lon max]
    ax.coastlines()
    ax.gridlines()
    plt.colorbar(
        im, ax=ax, ticks=levels, shrink=0.75, pad=0.04, label="Total precipitation (m)"
    )
    #plt.savefig("/home/jovyan/earth2studio-project/tp_outputs/02_tp_prediction.jpg")
    #plt.close(fig)

In [ ]:
ds = xr.open_zarr("/home/jovyan/earth2studio-project/tp_outputs/fcn_gfs_forecast.zarr")

extent=[220, 340, 20, 70]
plot_tp(ds, "tp", 8, extent)

# extent=[230, 250, 30, 45]
# plot_tp(ds, "tp", 8, extent)

## Hurricane Florence

uv add earth2studio --extra cyclone

uv add earth2studio --extra graphcast

### GraphCastOperational

In [ ]:
import os
import torch
from earth2studio.models.px import GraphCastOperational
from earth2studio.models.dx import TCTrackerWuDuan
from earth2studio.data import WB2ERA5
from earth2studio.io import ZarrBackend
from earth2studio.utils.time import to_time_array
from datetime import datetime, timedelta

os.chdir("/home/jovyan/earth2studio-project")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tracker = TCTrackerWuDuan()

package = GraphCastOperational.load_default_package()
graphcast_model = GraphCastOperational.load_model(package)
# prognostic = GraphCastOperational()

# data = ARCO()
data = WB2ERA5(cache=True, verbose=True)
os.makedirs("florence_outputs", exist_ok=True)

nsteps = 25
start_time = datetime(2018, 9, 13)
end_time = start_time + timedelta(hours=6 * nsteps)

print("Done")


In [ ]:
from tqdm import tqdm
from earth2studio.utils.coords import map_coords
from earth2studio.data import fetch_data

graphcast_operational_model = graphcast_model.to(device)

tracker.reset_path_buffer() #reset tracker

x, coords = fetch_data(
    source=data,
    time=to_time_array([start_time]),
    variable=graphcast_operational_model.input_coords()["variable"],
    lead_time=graphcast_operational_model.input_coords()["lead_time"],
    device=device
)

x, coords = map_coords(x, coords, graphcast_operational_model.input_coords())

model = graphcast_operational_model.create_iterator(x, coords)
with tqdm(total=nsteps+1, desc="Running inference") as pbar:
    for steps, (x, coords) in enumerate(model):
        x, coords = map_coords(x, coords, tracker.input_coords())
        output, output_coords = tracker(x, coords)
        output = output[:,0]
        print(f"Step {steps}: GraphCast tracker output shape {output.shape}")
        pbar.update(1)
        if steps == nsteps:
            break
graphcast_operational_tracks = output.cpu()
torch.save(graphcast_operational_tracks, "florence_outputs/graphcast_operational_paths.pt")
print("Saved Graphcast paths")

### Era5

In [ ]:
from earth2studio.data import fetch_data, prep_data_array, ARCO,WB2ERA5

# era5_data = ARCO()
era5_data = WB2ERA5(cache=True, verbose=True)
times = [start_time + timedelta(hours = 6 * i) for i in range(nsteps + 1)]

tracker.reset_path_buffer() #reset tracker

for step, time in enumerate(times):
    da = era5_data(time, tracker.input_coords()["variable"])
    x, coords = prep_data_array(da, device=device)
    output, output_coords = tracker(x, coords)
    print(f"Step {step}: WB2 tracks output shape {output.shape}")

era5_tracks = output.cpu()
torch.save(era5_tracks, "florence_outputs/era5_paths.pt")
print("Saved era5 track")

### FCN

In [ ]:
import os
import torch
from earth2studio.models.px import FCN
from earth2studio.models.dx import TCTrackerWuDuan
from earth2studio.data import NCAR_ERA5
from earth2studio.io import ZarrBackend
from earth2studio.utils.time import to_time_array
from datetime import datetime, timedelta


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tracker = TCTrackerWuDuan()
FCN_data = NCAR_ERA5()
package = FCN.load_default_package()
FCN_model = FCN.load_model(package)
start_time = datetime(2018, 9, 13)
print("Done")

In [ ]:
from tqdm import tqdm
from earth2studio.utils.coords import map_coords
from earth2studio.data import fetch_data
from earth2studio.utils.type import CoordSystem

from tqdm import tqdm
from earth2studio.utils.coords import map_coords

FCN_model = FCN_model.to(device)

tracker.reset_path_buffer() #reset tracker

x, coords = fetch_data(
    source=FCN_data,
    time=to_time_array([start_time]),
    variable=FCN_model.input_coords()["variable"],
    lead_time=FCN_model.input_coords()["lead_time"],
    device=device
)

x, coords = map_coords(
    x,
    coords,
    FCN_model.input_coords(),
)

model = FCN_model.create_iterator(x, coords)
with tqdm(total=nsteps+1, desc="Running inference") as pbar:
    for step, (x, coords) in enumerate(model):
        x, coords = map_coords(x, coords, tracker.input_coords())
        output, output_coords = tracker(x, coords)
        output = output[:,0]

        print(f"Step {step}: FCN tracker output shape {output.shape}")

        pbar.update(1)
        if step == nsteps:
            break
FCN_tracks = output.cpu()
torch.save(FCN_tracks, "florence_outputs/fcn_paths.pt")


## Era5 vs Graphcast vs FCN

### Helper Functions

In [ ]:
import matplotlib.patheffects as pe

#add city names for clearer plot
def add_cities(ax, city_list, default_offset=(0.22, 0.22),
               text_kw=None, marker_kw=None):
    if text_kw is None:
        text_kw = dict(fontsize=8, 
                       zorder=11,
                       path_effects=[pe.withStroke(linewidth=2, foreground="white")],
                       transform=ccrs.PlateCarree())
    if marker_kw is None:
        marker_kw = dict(marker="o",
                         color="red",
                         markersize=3, 
                         zorder=10,
                         transform=ccrs.PlateCarree())

    for item in city_list:
        # item can be (name, lat, lon) or (name, lat, lon, dx, dy)
        if len(item) == 3:
            name, lat, lon = item
            dx, dy = default_offset
        else:
            name, lat, lon, dx, dy = item

        ax.plot(lon, lat, **marker_kw)
        ax.text(lon + dx, lat + dy, name, **text_kw)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature


def plot_hurricane_tracks(
    models,
    start_time=None,
    end_time=None,
    extent=(-95, -60, 30, 45),
    cities=None,
    title="Tropical Cyclone Tracks",
    figsize=(10, 8),
    show=True,
):
    projection = ccrs.Robinson()

    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection=projection)

    # Map styling
    ax.add_feature(cfeature.LAND, alpha=0.1)
    ax.add_feature(cfeature.STATES, linewidth=0.5, alpha=0.5)
    #ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.gridlines(draw_labels=True, linewidth=0.6, alpha=0.2)
    ax.set_extent(extent)

    # Plot models
    for model in models:
        name = model["name"]
        tracks = model["tracks"]
        color = model.get("color", None)
        lw = model.get("lw", 2.0)
        ls = model.get("ls", "--")

        # torch → numpy
        if hasattr(tracks, "detach"):
            tracks = tracks.detach().cpu().numpy()

        label_done = False

        for p in range(tracks.shape[1]):
            lats = tracks[0, p, :, 0]
            lons = tracks[0, p, :, 1]
            mask = ~np.isnan(lats) & ~np.isnan(lons)

            if mask.any() and len(lons[mask]) > 2:
                ax.plot(
                    lons[mask],
                    lats[mask],
                    color=color,
                    linewidth=lw,
                    linestyle=ls,
                    label=name if not label_done else "",
                    transform=ccrs.PlateCarree(),
                    zorder=6,
                    # path_effects=[
                    #     pe.Stroke(linewidth=lw + 2.5, foreground="white"),
                    #     pe.Normal(),
    #],
                )
                label_done = True

    # Cities
    if cities is not None:
        add_cities(ax, cities)

    # Title
    if (start_time is not None) and (end_time is not None):
        ax.set_title(f"{title}\n{start_time:%Y-%m-%d} to {end_time:%Y-%m-%d}")
    else:
        ax.set_title(title)

    ax.legend(loc="lower right", title="Models")

    #plt.savefig("Florence(9-18).png", bbox_inches="tight")
    if show:
        plt.show()

    return fig, ax


In [ ]:

#load saved model paths
era5_load = torch.load("florence_outputs/era5_paths.pt", map_location="cpu")
graphcast_operational_load = torch.load("florence_outputs/graphcast_operational_paths.pt", map_location="cpu")
FCN_load = torch.load("florence_outputs/fcn_paths.pt", map_location="cpu")

# era5_paths = era5_load.numpy()
# graphcast_operational_paths = graphcast_operational_load.numpy()
# fcn_paths = FCN_load.numpy()

#list of cities to plot
cities = [
    # North Carolina
    ("Charlotte", 35.2271, -80.8431,  0.18, -0.18),
    ("Raleigh", 35.7796, -78.6382,  0.18,  0.18),

    # South Carolina
    ("Charleston", 32.7765, -79.9311, -0.35,  0.15),

    # Other major cities
    ("Atlanta", 33.7490, -84.3880,  0.18,  0.18),
    ("Richmond", 37.5407, -77.4360,  0.18, -0.18),
    ("Philadelphia", 39.9526, -75.1652,  0.18, -0.18),
    ("New York City", 40.7685, -73.9822,  0.18,  -0.18),
]

models = [
    {
        "name": "ERA5",
        "tracks": era5_load,
        "color": "#1f77b4",
        "lw": 3.0,
        "ls": "-"
    },
    {
        "name": "Graphcast Operational",
        "tracks": graphcast_operational_load,
        "color": "#ff7f0e",
        "lw": 2.0,
        "ls": "--"
    },
    {
        "name": "FCN",
        "tracks": FCN_load,
        "color": "#2ca02c",
        "lw": 2.0,
        "ls": "-."
    },
]
extent = (-90, -65, 31, 41)
plot_hurricane_tracks(
    models,
    start_time=start_time,
    end_time=end_time,
    extent=extent,
    cities=cities,
)


## Wind Visualizations

In [ ]:
import os
from datetime import datetime
import torch
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import earth2studio.run as run
from earth2studio.models.px import GraphCastOperational
from earth2studio.data import GFS
from earth2studio.io import ZarrBackend

os.chdir("/home/jovyan/earth2studio-project")
os.makedirs("outputs/wind_outputs", exist_ok=True)

In [ ]:
package = GraphCastOperational.load_default_package()
model = GraphCastOperational.load_model(package)
data = GFS()
io = ZarrBackend(
    "outputs/wind_outputs/graphcast_wind_forecast.zarr",
    backend_kwargs={"overwrite": True}
)

nsteps = 16  #4 days
io = run.deterministic(["2024-09-26"], nsteps, model, data, io)
print(io.root.tree())

In [ ]:
ds = xr.open_zarr("outputs/wind_outputs/graphcast_wind_forecast.zarr")
print(ds)
print("\nu10m shape:", ds["u10m"].shape)
print("Dimensions:", ds["u10m"].dims)

In [ ]:
def compute_wind_speed(ds, step):
    """
    Combine u10m and v10m into scalar wind speed.
    
    ds["u10m"] shape: (time, lead_time, lat, lon)
      - time index 0   → the single forecast start (2024-09-26)
      - lead_time step → 0=analysis, 1=+6h, 2=+12h ... 16=+96h
    
    u10m = eastward wind component (m/s), positive = blowing east
    v10m = northward wind component (m/s), positive = blowing north
    speed = sqrt(u² + v²) — scalar magnitude in m/s
    """
    u = ds["u10m"].isel(time=0, lead_time=step).values  # shape: (721, 1440)
    v = ds["v10m"].isel(time=0, lead_time=step).values
    speed = np.sqrt(u**2 + v**2)
    return u, v, speed

In [ ]:
def plot_wind_field(ds, step, extent=None, title=None,
                    quiver_stride=20, save_path=None):

    u, v, speed = compute_wind_speed(ds, step)
    lats = ds["lat"].values   # (-90 → 90)
    lons = ds["lon"].values   # (0 → 359.75)
    lead_hours = step * 6

    fig, ax = plt.subplots(
        figsize=(14, 8),
        subplot_kw={"projection": ccrs.PlateCarree()}
    )

    # Filled contour — wind speed magnitude
    levels = np.arange(0, 35, 2)
    cf = ax.contourf(
        lons, lats, speed,
        levels=levels,
        cmap="YlOrRd",
        transform=ccrs.PlateCarree(),
        extend="neither"
    )
    cbar = plt.colorbar(cf, ax=ax, orientation="horizontal",
                        pad=0.04, shrink=0.7, label="Wind Speed (m/s)")
    cbar.set_ticks(np.arange(0, 36, 4))

    # Quiver arrows — subsample so arrows don't overlap
    lon_2d, lat_2d = np.meshgrid(lons, lats)
    s = quiver_stride
    ax.quiver(
        lon_2d[::s, ::s],
        lat_2d[::s, ::s],
        u[::s, ::s],
        v[::s, ::s],
        transform=ccrs.PlateCarree(),
        scale=500,
        width=0.002,
        headwidth=4,
        color="black",
        alpha=0.6,
        zorder=5
    )

    # Map features
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.6)
    ax.add_feature(cfeature.STATES, linewidth=0.3, alpha=0.4)
    ax.add_feature(cfeature.LAND, facecolor="lightgray", alpha=0.2)
    ax.gridlines(draw_labels=True, linewidth=0.4, alpha=0.4,
                 xlocs=range(-180, 181, 30), ylocs=range(-90, 91, 15))

    if extent:
        ax.set_extent(extent, crs=ccrs.PlateCarree())

    if title is None:
        title = f"GraphCast 10m Wind Field — Lead time: +{lead_hours}h (2024-09-26)"
    ax.set_title(title, fontsize=13, pad=10)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        #print(f"Saved: {save_path}")
    plt.show()

In [ ]:
# times = [0, 4, 8, 16]
times = [0, 4]
for step in times:
    plot_wind_field(
        ds,
        step=step,
        extent=None,
        quiver_stride=50,
        save_path=f"outputs/wind_outputs/images/global_wind_step{step:02d}.jpg"
    )

In [ ]:
#Florida
plot_wind_field(
    ds,
    step=4,           # +24h forecast
    extent=[-100, -70, 20, 40],
    quiver_stride=10,  # denser arrows for regional zoom
    title="GraphCast 10m Wind — +24h — Gulf Coast (Hurricane Helene, Sept 2024)",
    save_path="outputs/wind_outputs/images/gulf_coast_wind_step04.jpg"
)


'''
Notes:
-temperature as color, pressure as contour (or vice versa)
-color bar (reduce length, make more narrow)
'''

## MSL + Temperature Overlay

This section adds **Mean Sea Level Pressure (MSL)** as black contour lines on top of the T2M color fill — a standard synoptic analysis chart used by every major operational weather center.

This reuses the T2M zarr you already ran — no new inference needed. Both `t2m` and `msl` are native FCN output variables, so they are already in the store.

**Why this combination matters:**  
Low pressure centers cluster at the cores of storm systems. Cold air masses trail behind cold fronts, warm air sits ahead of warm fronts. Overlaying both variables on one map lets you see the temperature-pressure relationship driving mid-latitude weather at a glance — something neither variable conveys alone.

### Cell: Imports

`scipy.ndimage.gaussian_filter` is the only new import. At 0.25-degree resolution, the raw MSL field has small-scale noise that makes contour lines unnecessarily jagged. A light Gaussian blur (sigma=1.5 grid cells) smooths this without moving the synoptic-scale features.

In [ ]:
from scipy.ndimage import gaussian_filter
import os

os.makedirs("outputs/overlay_outputs", exist_ok=True)

### Cell: Load data

Opens the zarr you already produced in the T2M section above. Both `t2m` and `msl` are present because FCN outputs all 26 of its native variables to the same store. The print statements confirm both variables exist and show their shapes before we try to plot.

In [ ]:
ds_overlay = xr.open_zarr("outputs/t2m_outputs/fcn_gfs_forecast.zarr")

lats_ov = ds_overlay["lat"].values
lons_ov = ds_overlay["lon"].values

print("Variables in store:", list(ds_overlay.data_vars))
print("t2m shape:", ds_overlay["t2m"].shape)
print("msl shape:", ds_overlay["msl"].shape)

### Cell: Plotting function

**Temperature fill:**  
`.isel(time=0, lead_time=step)` selects the single forecast start and a specific lead-time step. Temperature is converted from Kelvin to Celsius (subtract 273.15). `pcolormesh` fills each grid cell with color from the `RdBu_r` diverging colormap — red for warm, blue for cold, centered on 0°C.

**Pressure contours:**  
MSL comes out of the model in Pascals. We divide by 100 to get hPa, apply `gaussian_filter(sigma=1.5)` to smooth the field, then compute contour levels dynamically snapped to the nearest 4 hPa multiple. 4 hPa is the standard synoptic contour interval. `ax.clabel` labels every other line (every 8 hPa) so labels don't overlap.

**Projection:**  
Robinson is the display projection. `transform=ccrs.PlateCarree()` on both the `pcolormesh` and `contour` calls is required — it tells cartopy that the data coordinates are regular lat/lon degrees, separate from the map display projection.

In [ ]:
def plot_t2m_msl(ds, step, save_path=None):
    lead_hours = step * 6

    t2m_c = ds["t2m"].isel(time=0, lead_time=step).values - 273.15

    msl_hpa = gaussian_filter(ds["msl"].isel(time=0, lead_time=step).values / 100.0, sigma=1.5)

    fig, ax = plt.subplots(figsize=(14, 7), subplot_kw={"projection": ccrs.Robinson()})

    cf = ax.pcolormesh(
        lons_ov, lats_ov, t2m_c,
        transform=ccrs.PlateCarree(),
        cmap="RdBu_r", vmin=-40, vmax=40, shading="auto",
    )
    cbar = plt.colorbar(cf, ax=ax, orientation="horizontal", pad=0.04, shrink=0.6, aspect=40)
    cbar.set_label("2m Temperature (°C)", fontsize=18)

    p_min = int(np.floor(msl_hpa.min() / 4) * 4)
    p_max = int(np.ceil(msl_hpa.max() / 4) * 4)
    levels_msl = np.arange(p_min, p_max + 4, 4)

    cs = ax.contour(
        lons_ov, lats_ov, msl_hpa,
        levels=levels_msl, colors="black", linewidths=0.7,
        transform=ccrs.PlateCarree(),
    )
    ax.clabel(cs, levels=levels_msl[::2], inline=True, fontsize=14, fmt="%d", inline_spacing=4)

    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.5)
    ax.gridlines(linewidth=0.3, alpha=0.4, color="gray")

    ax.set_title(
        f"FCN — 2m Temperature (fill) & Mean Sea Level Pressure (contours)\n"
        f"Init: 2026-01-08  |  Lead: +{lead_hours}h",
        fontsize=22, pad=10
    )
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()

### Cell: Generate plots at +0h, +24h, +48h

Step 0 is the GFS initial condition — the actual observed atmospheric state. Steps 4 and 8 show how the pressure pattern propagates as the forecast advances.

**What to look for:**  
- Low pressure centers (closely packed isobars with lower values) sit at storm cores
- Cold air behind a cold front (blue fill) typically pairs with a trough (curved isobars bowing equatorward)
- Warm air ahead of a warm front (red fill) sits under a ridge (isobars bowing poleward)

In [ ]:
for step in [0]:
    plot_t2m_msl(
        ds_overlay,
        step=step,
        save_path=f"outputs/overlay_outputs/t2m_msl_step{step:02d}.jpg",
    )

## 500 hPa Geopotential Height

This section runs new FCN inference and visualizes **500 hPa geopotential height** — the single most important variable in synoptic meteorology.

**What z500 is:**  
The 500 hPa pressure level is the altitude where exactly half the atmospheric mass is above you and half is below — roughly 5.5 km up. The height of that surface varies because warmer air is less dense and expands upward. Where the 500 hPa surface is high (a *ridge*), the underlying air column is warm. Where it is low (a *trough*), the column is cold.

**Why it drives surface weather:**  
The horizontal pattern of z500 determines mid-tropospheric wind flow, which steers surface weather systems. Troughs bring cold air southward and are associated with surface lows and active weather. Ridges pump warm air northward and suppress convection. NWS forecasters call 500 hPa the "steering level."

**Visualization convention:**  
- **Color fill:** anomaly from the zonal (latitude-circle) mean — positive = ridge, negative = trough
- **Contour lines:** absolute geopotential height every 60 m — the standard NWS/ECMWF contour interval
- **Projection:** North Polar Stereographic — the standard operational upper-level analysis view

### Cell: Run inference

Same FCN + GFS pattern as the T2M section. Same start date (2026-01-08) so the z500 output is directly comparable to the T2M maps you already have. `z500` is a native FCN variable, so no diagnostic model is needed.

After the run, `io.root.tree()` prints the zarr structure confirming z500 is present.

In [ ]:
import os

z500_output_path = "outputs/z500_outputs"

os.makedirs(z500_output_path, exist_ok=True)

model_z500 = FCN.load_model(FCN.load_default_package())
data_z500 = GFS()
io_z500 = ZarrBackend(f"{z500_output_path}/fcn_gfs_z500.zarr", backend_kwargs={"overwrite": True})

io_z500 = run.deterministic(["2026-01-08"], 10, model_z500, data_z500, io_z500)
print(io_z500.root.tree())

### Cell: Load output and define helper

`xr.open_zarr` opens the store lazily. We also define `get_z500_fields` here, which handles two things:

**Unit conversion:**  
The model stores z500 in m²/s² (geopotential). Dividing by `g = 9.80665 m/s²` gives geopotential height in meters — the form used in all standard weather products. For z500, this converts values around 55,000 m²/s² to roughly 5,600 m.

**Zonal anomaly:**  
`z_smooth.mean(axis=1, keepdims=True)` averages over longitudes at each latitude (axis=1 is the longitude axis for a `(lat, lon)` array), giving the latitude-circle mean height. Subtracting it removes the large background gradient between tropics and poles, so the ridge/trough pattern stands out clearly with a colormap centered on zero. The light `gaussian_filter(sigma=1.0)` smooths gridscale noise before contouring.

In [ ]:
ds_z500 = xr.open_zarr("z500_outputs/fcn_gfs_z500.zarr")
lats_z500 = ds_z500["lat"].values
lons_z500 = ds_z500["lon"].values

print("z500 shape:", ds_z500["z500"].shape)

G = 9.80665

def get_z500_fields(ds, step):
    z_raw = ds["z500"].isel(time=0, lead_time=step).values
    z_m = gaussian_filter(z_raw / G, sigma=1.0)             # geopotential → height in meters, smoothed
    anomaly = z_m - z_m.mean(axis=1, keepdims=True)         # subtract zonal mean at each latitude
    return z_m, anomaly

### Cell: Plotting function

**Projection — NorthPolarStereo:**  
`ccrs.NorthPolarStereo(central_longitude=260)` gives a top-down view of the Northern Hemisphere centered over North America. This is the standard projection for upper-level analysis at operational forecast centers. `ax.set_extent([-180, 180, 20, 90])` limits the view to latitudes above 20°N.

**Anomaly fill:**  
`np.linspace(-150, 150, 31)` gives 31 levels spanning ±150 m — enough to capture the full range of synoptic-scale ridge/trough amplitude. `extend="both"` keeps extreme colors for values beyond the range rather than clipping.

**Absolute height contours:**  
Levels are snapped to the nearest 60 m multiple so they never land on odd values. Every other line is labeled (every 120 m) to prevent crowding.

In [ ]:
def plot_z500(ds, step, save_path=None):
    lead_hours = step * 6
    z_abs, z_anom = get_z500_fields(ds, step)

    fig, ax = plt.subplots(
        figsize=(10, 10),
        subplot_kw={"projection": ccrs.NorthPolarStereo(central_longitude=260)}
    )
    ax.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())

    cf = ax.contourf(
        lons_z500, lats_z500, z_anom,
        levels=np.linspace(-150, 150, 31),
        transform=ccrs.PlateCarree(),
        cmap="RdBu_r", extend="both",
    )
    cbar = plt.colorbar(cf, ax=ax, orientation="horizontal", pad=0.04, shrink=0.7, aspect=30)
    cbar.set_label("Z500 Anomaly from Zonal Mean (m)", fontsize=10)
    cbar.set_ticks(np.arange(-150, 151, 50))

    z_min = int(np.floor(z_abs.min() / 60) * 60)
    z_max = int(np.ceil(z_abs.max() / 60) * 60)
    contour_levels = np.arange(z_min, z_max + 60, 60)
    cs = ax.contour(
        lons_z500, lats_z500, z_abs,
        levels=contour_levels, colors="black", linewidths=0.7,
        transform=ccrs.PlateCarree(),
    )
    ax.clabel(cs, inline=True, fontsize=7, fmt="%d", inline_spacing=4,
              levels=contour_levels[::2])

    ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.5)
    ax.gridlines(linewidth=0.4, alpha=0.4, color="gray",
                 xlocs=range(-180, 181, 30), ylocs=range(20, 91, 10))

    ax.set_title(
        f"FCN — 500 hPa Geopotential Height\n"
        f"Fill: Zonal anomaly  |  Contours: Absolute height (60 m interval)\n"
        f"Init: 2026-01-08  |  Lead: +{lead_hours}h",
        fontsize=11, pad=10
    )
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

### Cell: Generate plots

Four lead times spanning the full forecast window. Compare the anomaly pattern here directly to the T2M map from the first section — cold surface anomalies align with blue-filled troughs, warm anomalies with red-filled ridges. This is the physical connection between the upper-level steering flow and the surface temperature distribution.

In [ ]:
for step in [0, 4, 8, 10]:
    plot_z500(
        ds_z500,
        step=step,
        save_path=f"z500_outputs/z500_step{step:02d}.jpg",
    )

## Wind Gust Forecast (WindgustAFNO)

This section uses the `WindgustAFNO` diagnostic model to predict surface wind gusts — a variable not produced directly by any of the prognostic models in this notebook.

**Install before running:**
```
uv add earth2studio --extra windgust-afno
```

**Gusts vs. sustained wind:**  
The `u10m` and `v10m` variables FCN outputs are *sustained* 10-meter winds — the mean wind over some interval. Wind gusts are the *peak instantaneous* speed within a 6-hour window, consistently 20–50% higher. Every NWS wind hazard warning, aviation ground stop, and building damage threshold is based on gust speed, not sustained wind. The sustained wind tells you the background flow; the gust tells you the worst-case impact.

**Pattern:**  
Identical to the Total Precipitation section — load a prognostic model, load a diagnostic model, pass both to `run.diagnostic`. The diagnostic is called after every FCN time step to generate the gust field for that 6-hour window.

**Three visualizations:**
1. Global gust map with Beaufort-scale hazard contours
2. CONUS regional zoom
3. Gust *excess* (gust minus sustained wind) — reveals where boundary layer turbulence and convective mixing are strongest

### Cell: Load models and run inference

`WindgustAFNO` slots in where `PrecipitationAFNO` was in the precipitation section — the `run.diagnostic` call is structurally identical. January 8, 2024 is a winter storm case with high gust activity over the central and eastern US. Change the date to any event of interest.

In [ ]:
from earth2studio.models.dx import WindgustAFNO
import os

windgust_output_path = "outputs/windgust_outputs"
os.makedirs(windgust_output_path,exist_ok=True)

prognostic_wg = FCN.load_model(FCN.load_default_package())
diagnostic_wg = WindgustAFNO.load_model(WindgustAFNO.load_default_package())

data_wg = GFS()
io_wg = ZarrBackend(f"{windgust_output_path}/windgust_outputs/fcn_windgust_forecast.zarr", backend_kwargs={"overwrite": True})

io_wg = run.diagnostic(
    ["2024-01-08"], 10,
    prognostic_wg, diagnostic_wg,
    data_wg, io_wg,
)
print(io_wg.root.tree())

### Cell: Load output

The gust field is stored as `fg10` — "wind gust at 10 meters" — following ECMWF variable naming convention. Print the variable list from `io.root.tree()` above to confirm the name before running this cell.

In [ ]:
ds_wg = xr.open_zarr("windgust_outputs/fcn_windgust_forecast.zarr")
lats_wg = ds_wg["lat"].values
lons_wg = ds_wg["lon"].values

print("Variables:", list(ds_wg.data_vars))

gust_var = "fg10"
print(f"Gust shape: {ds_wg[gust_var].shape}")

### Cell: Gust map with hazard thresholds

**Color scale:** `plasma_r` (reversed) is perceptually uniform — every step in color equals the same step in data value. Light = calm, deep purple = extreme. `extend="max"` keeps values above 30 m/s dark rather than clipping.

**Hazard thresholds** drawn as dashed colored lines at WMO Beaufort scale boundaries:
- **17.2 m/s (amber)** — Beaufort 8, gale force. Tree branches break; NWS High Wind Advisory threshold
- **24.5 m/s (red)** — Beaufort 10, storm force. Trees uprooted, structural damage begins
- **32.7 m/s (purple)** — Beaufort 12, hurricane force. NWS Extreme Wind Warning threshold

The legend uses empty `ax.plot([], [])` lines as proxy artists — `ax.contour` does not generate legend entries automatically, so this is the standard workaround.

In [ ]:
GALE_MS      = 17.2
STORM_MS     = 24.5
HURRICANE_MS = 32.7

def plot_windgust(ds, step, extent=None, save_path=None):
    lead_hours = step * 6
    gust = ds[gust_var].isel(time=0, lead_time=step).values

    proj = ccrs.Robinson() if extent is None else ccrs.PlateCarree()
    fig, ax = plt.subplots(figsize=(13, 7), subplot_kw={"projection": proj})

    cf = ax.contourf(
        lons_wg, lats_wg, gust,
        levels=np.arange(0, 32, 2),
        transform=ccrs.PlateCarree(),
        cmap="plasma_r", extend="max",
    )
    cbar = plt.colorbar(cf, ax=ax, orientation="horizontal", pad=0.04, shrink=0.75, aspect=40)
    cbar.set_label("10m Wind Gust (m/s)", fontsize=11)
    cbar.set_ticks([0, 5, 10, 15, 20, 25, 30])

    for thresh, color, label in zip(
        [GALE_MS,   STORM_MS,  HURRICANE_MS],
        ["#f59e0b", "#ef4444", "#7c3aed"],
        ["Gale (17 m/s)", "Storm (25 m/s)", "Hurricane (33 m/s)"]
    ):
        ax.contour(lons_wg, lats_wg, gust, levels=[thresh],
                   colors=[color], linewidths=1.8, linestyles="--",
                   transform=ccrs.PlateCarree())
        ax.plot([], [], color=color, linewidth=1.8, linestyle="--", label=label)

    ax.legend(loc="lower left", fontsize=9, framealpha=0.85,
              title="Beaufort Hazard Thresholds", title_fontsize=9)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.5)
    ax.add_feature(cfeature.STATES, linewidth=0.3, alpha=0.4)
    ax.gridlines(linewidth=0.3, alpha=0.3, color="gray")

    if extent:
        ax.set_extent(extent, crs=ccrs.PlateCarree())

    ax.set_title(
        f"FCN + WindgustAFNO — 10m Wind Gust\nInit: 2024-01-08  |  Lead: +{lead_hours}h",
        fontsize=12, pad=10
    )
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

# Global overview
for step in [4, 8]:
    plot_windgust(ds_wg, step, save_path=f"windgust_outputs/global_gust_step{step:02d}.jpg")

### Cell: CONUS zoom

Zooms to `[-130, -60, 20, 55]` at +24h. The addition of state boundary lines (`cfeature.STATES`) makes it possible to identify which NWS warning zones fall inside each hazard contour — the format operational forecasters actually use.

In [ ]:
plot_windgust(
    ds_wg, step=4,
    extent=[-130, -60, 20, 55],
    save_path="windgust_outputs/conus_gust_step04.jpg",
)

### Cell: Gust excess (gust minus sustained wind)

The difference between the diagnostic gust and the prognostic sustained wind is a measure of boundary layer turbulence intensity.

High gust excess (8–12 m/s above sustained) indicates:
- Active convection — thunderstorm downdrafts dramatically amplify gusts
- Strong daytime heating — unstable boundary layers over land mix higher winds downward
- Orographic turbulence — mechanical mixing over terrain

Low gust excess (1–3 m/s) indicates stable, laminar flow — typical over calm ocean surfaces or under strong inversions.

`np.clip(..., 0, None)` removes small negative values that appear where the diagnostic slightly underestimates relative to the prognostic sustained wind. These are model artifacts with no physical meaning.

In [ ]:
def plot_gust_excess(ds, step, save_path=None):
    lead_hours = step * 6
    gust      = ds[gust_var].isel(time=0, lead_time=step).values
    sustained = np.sqrt(ds["u10m"].isel(time=0, lead_time=step).values**2 +
                        ds["v10m"].isel(time=0, lead_time=step).values**2)
    excess    = np.clip(gust - sustained, 0, None)

    fig, ax = plt.subplots(figsize=(13, 7), subplot_kw={"projection": ccrs.Robinson()})

    cf = ax.contourf(
        lons_wg, lats_wg, excess,
        levels=np.arange(0, 14, 1),
        transform=ccrs.PlateCarree(),
        cmap="OrRd", extend="max",
    )
    cbar = plt.colorbar(cf, ax=ax, orientation="horizontal", pad=0.04, shrink=0.75, aspect=40)
    cbar.set_label("Gust Excess above Sustained Wind (m/s)", fontsize=11)

    ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.5)
    ax.gridlines(linewidth=0.3, alpha=0.3, color="gray")

    ax.set_title(
        f"Wind Gust Excess (Gust − Sustained 10m Wind)\nInit: 2024-01-08  |  Lead: +{lead_hours}h",
        fontsize=12, pad=10
    )
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

plot_gust_excess(ds_wg, step=4, save_path="windgust_outputs/gust_excess_step04.jpg")

## Forecast Animations

This section animates both the T2M and precipitation forecasts using `matplotlib.animation.FuncAnimation` — the standard matplotlib approach for building frame-by-frame videos. Both animations reuse the zarr files already on disk. No new inference needed.

**Why animations over static plots:**  
A static plot shows one moment. An animation shows *propagation* — you can watch fronts advance, storm systems intensify, and moisture plumes shift in real time. At 6-hour intervals across 10 steps, these animations cover 60 hours of forecast evolution in a few seconds of video.

**Saving requires ffmpeg on PATH.** On the TIDE cluster this is available by default. Both animations are saved as MP4 files.

### Cell: Animation imports

In [ ]:
import matplotlib.animation as animation

os.makedirs("animation_outputs", exist_ok=True)

### Cell: T2M animation

**How `FuncAnimation` works:**  
You build the figure once with all static elements — map features, colorbar, axes — then define an `update(step)` function that changes only what needs to change per frame. `FuncAnimation` calls `update` once per frame index and collects the results into a video.

**Critical: fixed color bounds.**  
`vmin=-50, vmax=50` are constant across all frames. If the scale changed per frame, the same color would represent different temperatures in different frames — visually misleading and scientifically wrong.

**`mesh.set_array(t2m_all[step].ravel())`** replaces the pcolormesh data in-place without re-drawing the entire artist. This is fast and is what `blit=True` relies on — blit mode only re-renders the artists that `update` returns, skipping everything static.

The full T2M array is loaded into memory at once (`ds["t2m"].isel(time=0).values`) rather than reading one step at a time. For a 721×1440 grid with 11 steps this is ~45 MB — well within limits and much faster than repeated disk reads during animation.

In [ ]:
ds_t2m_anim = xr.open_zarr("/home/jovyan/earth2studio-project/t2m_outputs/fcn_gfs_forecast.zarr")

def make_t2m_animation(ds, save_path):
    lats = ds["lat"].values
    lons = ds["lon"].values
    t2m_all = ds["t2m"].isel(time=0).values - 273.15  # full array: (lead_time, lat, lon)

    fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={"projection": ccrs.Robinson()})
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.5)
    ax.gridlines(linewidth=0.3, alpha=0.3, color="gray")

    mesh = ax.pcolormesh(
        lons, lats, t2m_all[0],
        transform=ccrs.PlateCarree(),
        cmap="RdBu_r", vmin=-50, vmax=50, shading="auto",
    )
    cbar = plt.colorbar(mesh, ax=ax, orientation="horizontal", pad=0.04, shrink=0.75, aspect=40)
    cbar.set_label("2m Temperature (°C)", fontsize=10)
    title = ax.set_title("", fontsize=11)

    def update(step):
        mesh.set_array(t2m_all[step].ravel())
        title.set_text(f"FCN — 2m Temperature  |  Init: 2026-01-08  |  Lead: +{step*6}h")
        return mesh, title

    anim = animation.FuncAnimation(fig, update, frames=t2m_all.shape[0], interval=350, blit=True)
    anim.save(save_path, writer=animation.FFMpegWriter(fps=3, bitrate=1800), dpi=120)
    plt.close(fig)
    print(f"Saved: {save_path}")

make_t2m_animation(ds_t2m_anim, "animation_outputs/t2m_animation.mp4")

### Cell: Precipitation animation (North America zoom)

Precipitation uses `contourf` instead of `pcolormesh`, which creates a different animation challenge. `pcolormesh` returns a single `QuadMesh` whose underlying data can be swapped with `.set_array()`. `contourf` returns a `QuadContourSet` — a collection of polygon patches — and there is no equivalent in-place update. The entire set of polygons must be removed and redrawn each frame.

The `update` function loops over `ax.collections` and calls `.remove()` on each to clear the previous frame, then calls `contourf` again. This is slower, which is why `blit=False` is used — blit requires modifying only known artists, but here we are clearing and replacing the whole collection list.

**Units:** model outputs precipitation in meters, multiplied by 1000 here to convert to millimeters — the standard unit in weather products.

**Extent:** `[220, 310, 15, 75]` is North America in 0–360 longitude convention (220 = 140°W, 310 = 50°W).

In [ ]:
ds_tp_anim = xr.open_zarr("/home/jovyan/earth2studio-project/tp_outputs/fcn_gfs_forecast.zarr")

def make_tp_animation(ds, save_path):
    lats = ds["lat"].values
    lons = ds["lon"].values
    tp_all = ds["tp"].isel(time=0).values * 1000  # meters → millimeters: (lead_time, lat, lon)

    extent = [220, 310, 15, 75]
    levels = np.linspace(0, 8, 17)

    fig, ax = plt.subplots(figsize=(10, 7), subplot_kw={"projection": ccrs.PlateCarree()})
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
    ax.add_feature(cfeature.STATES, linewidth=0.4, alpha=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.add_feature(cfeature.LAND, facecolor="whitesmoke", zorder=0)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, color="gray")
    gl.top_labels = False
    gl.right_labels = False

    cf = ax.contourf(lons, lats, tp_all[0], levels=levels,
                     transform=ccrs.PlateCarree(), cmap="YlGnBu", extend="max")
    cbar = plt.colorbar(cf, ax=ax, orientation="vertical", pad=0.02, shrink=0.85)
    cbar.set_label("Total Precipitation (mm)", fontsize=10)
    title = ax.set_title("", fontsize=11)

    def update(step):
        for coll in ax.collections:
            coll.remove()
        cf_new = ax.contourf(lons, lats, tp_all[step], levels=levels,
                             transform=ccrs.PlateCarree(), cmap="YlGnBu", extend="max")
        title.set_text(f"FCN + PrecipAFNO — Total Precipitation  |  Init: 2026-01-01  |  Lead: +{step*6}h")
        return cf_new.collections

    anim = animation.FuncAnimation(fig, update, frames=tp_all.shape[0], interval=400, blit=False)
    anim.save(save_path, writer=animation.FFMpegWriter(fps=2, bitrate=1800), dpi=120)
    plt.close(fig)
    print(f"Saved: {save_path}")

make_tp_animation(ds_tp_anim, "animation_outputs/tp_animation.mp4")